# ADHD-200 (AAL-116) FC Extraction + Baseline Classification

Pipeline:
1. Read `<SITE>_phenotypic.csv` for every site -> subject DX labels.
2. For each subject folder, load **all** `*_aal_TCs.1D` runs.
3. Compute a 116x116 Pearson FC matrix per run 
4. Average FC matrices across runs -> one FC matrix per subject.
5. Report subject counts (per site, per DX class).
6. Build a feature matrix (upper triangle of FC) + labels.
7. Run baseline classification (ADHD vs Control) with 5-fold CV.

**DX key:** 0 = Control, 1 = ADHD-Combined, 2 = ADHD-Hyperactive/Impulsive, 3 = ADHD-Inattentive


## 0. Setup

In [27]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")  # GPU 1 becomes cuda:0 now

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


In [28]:
from pathlib import Path
import pandas as pd
import glob
import os
import warnings

import numpy as np
import pandas as pd

N_ROIS = 116  # AAL-116 atlas


SITES = [
    "KKI", "NYU", "Peking_1","Peking_2","Peking_3", "OHSU", "NeuroIMAGE",
    "WashU", "Pittsburgh",
]

ROOT = Path(r"/home/siu856622573/Code/ADHD code/preprocessing")
print("Root exists:", os.path.isdir(ROOT))

DATA = Path(r"/home/siu856622573/Code/ADHD200_AAL_TCs_filtfix (1)")
site_summary = []
total_subjects = 0

for site in sorted(DATA.iterdir()):

    if not site.is_dir():
        continue

    if site.name == "templates":
        continue

    subjects = [d for d in site.iterdir() if d.is_dir()]

    site_summary.append({
        "Site": site.name,
        "Subjects": len(subjects)
    })

    total_subjects += len(subjects)

summary = pd.DataFrame(site_summary)

print(summary)

print("\nTotal Sites :", len(summary))
print("Total Subjects :", total_subjects)


Root exists: True
         Site  Subjects
0         KKI        83
1         NYU       222
2  NeuroIMAGE        48
3        OHSU        79
4    Peking_1        85
5    Peking_2        67
6    Peking_3        42
7  Pittsburgh        89
8       WashU        61

Total Sites : 9
Total Subjects : 776


## 1. Phenotypic loading

In [29]:
def load_phenotypic(site_dir, site_name):
    """Find and load the <SITE>_phenotypic.csv for a site directory.
    Returns a DataFrame indexed by a *string* subject id (as found in the CSV,
    without assuming any particular zero-padding)."""
    candidates = glob.glob(os.path.join(site_dir, "*phenotypic*.csv"))
    if not candidates:
        warnings.warn(f"No phenotypic CSV found for site {site_name} in {site_dir}")
        return None

    csv_path = candidates[0]
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    id_col = None
    for c in df.columns:
        if c.strip().lower() in ("scandir id", "scandirid", "subject", "subid", "id"):
            id_col = c
            break
    if id_col is None:
        raise ValueError(f"Could not find a subject-ID column in {csv_path}. "
                          f"Columns found: {list(df.columns)}")

    df["ScanDirID_str"] = df[id_col].astype(str).str.strip()
    df = df.set_index("ScanDirID_str", drop=False)
    df["Site"] = site_name
    return df


def match_subject_dir(site_dir, subj_id_str):
    """CSV ScanDir ID may not be zero-padded the same way as the folder name
    (e.g. CSV has '10002', folder is '0010002'). Try a few zero-padding widths
    and, failing that, fuzzy-match on the numeric suffix."""
    direct = os.path.join(site_dir, subj_id_str)
    if os.path.isdir(direct):
        return direct

    numeric = "".join(ch for ch in subj_id_str if ch.isdigit())
    if numeric:
        for width in (7, 6, 8, 5):
            padded = numeric.zfill(width)
            candidate = os.path.join(site_dir, padded)
            if os.path.isdir(candidate):
                return candidate

    if numeric:
        for entry in os.listdir(site_dir):
            full = os.path.join(site_dir, entry)
            if os.path.isdir(full) and entry.lstrip("0") == numeric.lstrip("0") and entry.lstrip("0") != "":
                return full

    return None


## 2. Load `.1D` time series and compute FC matrices

In [30]:
def load_1d_timeseries(path):
    """Load one *_aal_TCs.1D file -> (n_timepoints, 116) numpy array.
    Handles files with or without a text header row."""
    with open(path, "r") as f:
        first_line = f.readline()

    has_header = any(ch.isalpha() for ch in first_line)

    df = pd.read_csv(path, sep=r"\s+", header=0 if has_header else None, engine="python")

    drop_cols = [c for c in df.columns if isinstance(c, str) and c.lower() in ("file", "sub-brick")]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=1, how="all")

    data = df.values.astype(float)

    if data.shape[1] != N_ROIS:
        if data.shape[1] == N_ROIS + 1:
            data = data[:, 1:]
        else:
            raise ValueError(f"{path}: expected {N_ROIS} ROI columns, got {data.shape[1]}")

    return data  # (timepoints, 116)


def compute_fc_matrix(ts):
    """ts: (timepoints, 116) -> 116x116 FC matrix."""
    ts = (ts - ts.mean(axis=0, keepdims=True)) / (ts.std(axis=0, keepdims=True) + 1e-8)
    fc = np.corrcoef(ts.T)
    np.fill_diagonal(fc, 1.0)
    return fc


def process_subject(subject_dir):
    """Load every *_aal_TCs.1D run in a subject folder, compute FC per run,
    average across runs. Returns (avg_fc, n_runs_used) or (None, 0) on failure."""
    run_files = sorted(glob.glob(os.path.join(subject_dir, "*_aal_TCs.1D")))
    if not run_files:
        return None, 0

    fc_list = []
    for rf in run_files:
        try:
            ts = load_1d_timeseries(rf)
            if ts.shape[0] < 10:
                continue
            fc = compute_fc_matrix(ts)
            fc_list.append(fc)
        except Exception as e:
            warnings.warn(f"Failed to process {rf}: {e}")

    if not fc_list:
        return None, 0

    avg_fc = np.mean(fc_list, axis=0)
    return avg_fc, len(fc_list)


## 3. Build the dataset (loops over all sites/subjects)

In [31]:
def upper_triangle_features(fc):
    iu = np.triu_indices(N_ROIS, k=1)
    return fc[iu]


def build_dataset(root, sites=SITES, verbose=True):
    records = []
    features = []

    # --- instrumentation: track why subjects get dropped ---
    skip_reasons = {"no_pheno_match": 0, "fc_failed": 0, "bad_dx": 0}
    skipped_subjects = []  # (site, subject_id_or_dirname, reason)

    for site in sites:
        site_dir = os.path.join(root, site)
        if not os.path.isdir(site_dir):
            if verbose:
                print(f"[skip] site dir not found: {site_dir}")
            continue

        pheno = load_phenotypic(site_dir, site)
        if pheno is None:
            continue

        n_site_subjects = 0
        for subj_id in pheno.index:
            subj_dir = match_subject_dir(site_dir, subj_id)
            if subj_dir is None:
                skip_reasons["no_pheno_match"] += 1
                skipped_subjects.append((site, subj_id, "no_pheno_match"))
                continue

            avg_fc, n_runs = process_subject(subj_dir)
            if avg_fc is None:
                skip_reasons["fc_failed"] += 1
                skipped_subjects.append((site, subj_id, "fc_failed"))
                continue

            row = pheno.loc[subj_id]
            dx = row.get("DX", np.nan)
            try:
                dx = int(dx)
            except (ValueError, TypeError):
                skip_reasons["bad_dx"] += 1
                skipped_subjects.append((site, subj_id, "bad_dx"))
                continue

            features.append(upper_triangle_features(avg_fc))
            records.append({
                "subject_id": subj_id,
                "site": site,
                "n_runs_used": n_runs,
                "DX": dx,
                "DX_binary": 0 if dx == 0 else 1,
                "Gender": row.get("Gender", np.nan),
                "Age": row.get("Age", np.nan),
            })
            n_site_subjects += 1

        if verbose:
            print(f"[{site}] usable subjects: {n_site_subjects}")

    if not records:
        raise RuntimeError("No usable subjects found. Check ROOT path and folder structure.")

    X = np.vstack(features)
    meta = pd.DataFrame(records)

    if verbose:
        print("\n--- Subject drop summary (pheno-listed but not in final X/meta) ---")
        print("By reason:", skip_reasons)
        print("Total dropped:", sum(skip_reasons.values()))
        if skipped_subjects:
            print("\nDetail:")
            for s in skipped_subjects:
                print(f"  site={s[0]:<12} subject_id={s[1]:<10} reason={s[2]}")

    # stash for inspection outside the function too
    build_dataset.last_skip_reasons = skip_reasons
    build_dataset.last_skipped_subjects = skipped_subjects

    return X, meta


In [32]:
CACHE_PATH = "fc_dataset_ADHD.npz"
SUMMARY_CSV = "subject_summary.csv"
REBUILD = False  # set True to force re-extraction

if os.path.exists(CACHE_PATH) and not REBUILD:
    print(f"Loading cached dataset from {CACHE_PATH} (set REBUILD=True to force re-extraction)")
    npz = np.load(CACHE_PATH, allow_pickle=True)
    X = npz["X"]
    meta = pd.DataFrame(npz["meta"].item())
else:
    print("Extracting FC matrices from raw .1D files (this can take a while)...")
    X, meta = build_dataset(DATA, sites=SITES)
    np.savez_compressed(CACHE_PATH, X=X, meta=meta.to_dict())
    meta.to_csv(SUMMARY_CSV, index=False)
    print(f"\nCached features -> {CACHE_PATH}")
    print(f"Subject summary -> {SUMMARY_CSV}")

print("Feature matrix shape:", X.shape)
meta.head()


Loading cached dataset from fc_dataset_ADHD.npz (set REBUILD=True to force re-extraction)
Feature matrix shape: (768, 6670)


,subject_id,site,n_runs_used,DX,DX_binary,Gender,Age
0,1018959,KKI,2,0,0,0.0,12.36
1,1019436,KKI,2,3,1,1.0,12.98
2,1043241,KKI,2,0,0,1.0,9.12
3,1266183,KKI,2,0,0,0.0,9.67
4,1535233,KKI,2,0,0,1.0,9.64


In [33]:
# Sanity check: are X and meta still row-aligned?
npz = np.load("fc_dataset_ADHD.npz", allow_pickle=True)
meta_from_cache = pd.DataFrame(npz["meta"].item())
print((meta.reset_index(drop=True)["subject_id"] == meta_from_cache["subject_id"]).all())
print(len(meta), X.shape[0])  # must match too
print(meta["n_runs_used"].value_counts())

True
768 768
n_runs_used
2     466
4     188
6     108
12      4
8       2
Name: count, dtype: int64


## 4. Subject counts

In [34]:
print("=== Subject counts ===")
print(meta["site"].value_counts())

print(f"\nTotal usable subjects: {len(meta)}")

print("\nBy binary label (0=Control,1=ADHD):")
print(meta["DX_binary"].value_counts())



=== Subject counts ===
site
NYU           216
Pittsburgh     89
Peking_1       85
KKI            83
OHSU           79
Peking_2       67
WashU          59
NeuroIMAGE     48
Peking_3       42
Name: count, dtype: int64

Total usable subjects: 768

By binary label (0=Control,1=ADHD):
DX_binary
0    488
1    280
Name: count, dtype: int64


### AAL ROI names 

Builds a 116-length list of real AAL region names (e.g. `Precentral_L`, `Frontal_Sup_R`), in the same ROI order used everywhere  and saved the mapping to a separate `aal_roi_names.json` file so it's reusable outside this notebook

In [35]:
import json
import os
import glob
from xml.etree import ElementTree
from sklearn.utils import Bunch
from nilearn import image as _image

AAL_LOCAL_DIR = "/home/siu856622573/Code/aal"

def load_local_aal(base_dir=AAL_LOCAL_DIR):
    """Load the AAL atlas from a local directory (instead of downloading from nilearn)."""
    nii_candidates = sorted(glob.glob(os.path.join(base_dir, "**", "*.nii"), recursive=True))
    xml_candidates = sorted(glob.glob(os.path.join(base_dir, "**", "*.xml"), recursive=True))

    nii_path = next((p for p in nii_candidates if "aal" in os.path.basename(p).lower()),
                     nii_candidates[0] if nii_candidates else None)
    xml_path = next((p for p in xml_candidates if "aal" in os.path.basename(p).lower()),
                     xml_candidates[0] if xml_candidates else None)

    if nii_path is None or xml_path is None:
        raise FileNotFoundError(
            f"Could not find AAL .nii/.xml under {base_dir}. "
            f"Found .nii: {nii_candidates}, .xml: {xml_candidates}"
        )

    labels = ["Background"]
    indices = ["0"]
    xml_tree = ElementTree.parse(xml_path)
    root = xml_tree.getroot()
    for label in root.iter("label"):
        indices.append(label.find("index").text)
        labels.append(label.find("name").text)

    return Bunch(maps=nii_path, labels=labels, indices=indices)


def build_aal_roi_names():
    """Returns a list of 116 AAL region names, ordered to match ROI_NETWORK / the FC matrix
    columns (ascending AAL label value -- same convention used in build_aal_to_yeo7 and the
    gradient-visualization cells)."""
    aal = load_local_aal()
    aal_img = _image.load_img(aal.maps)
    aal_data = aal_img.get_fdata()
    aal_values = np.unique(aal_data)
    aal_values = aal_values[aal_values != 0]

    label_lookup = {int(idx): lab for idx, lab in zip(aal.indices, aal.labels)}
    names = [label_lookup[int(v)] for v in aal_values]
    return names


ROI_NAMES = build_aal_roi_names()
assert len(ROI_NAMES) == N_ROIS

with open("aal_roi_names.json", "w") as f:
    json.dump(ROI_NAMES, f, indent=2)

print(f"Saved {len(ROI_NAMES)} ROI names -> aal_roi_names.json")


Saved 116 ROI names -> aal_roi_names.json


### Named edge list -> separate JSON file

Same idea as the `[i]:<weight>:[j]` edge lines inside the structure prompt, but with real AAL region names instead of numbers, saved on its own (not mixed into the text prompt), e.g. `Precentral_L:0.81:Frontal_Sup_R` 

In [36]:
# Verifies ROI_NAMES[i-1] actually corresponds to FC matrix column i, by
# comparing it against the REAL column header of one subject's .1D file --
# not just trusting that atlas-code sorting lines up.

import re

def get_1d_header_codes(subj_dir):
    """Finds a .1D file in subj_dir and extracts the ordered list of AAL
    codes from its header row (e.g. 'Mean_2001' -> 2001)."""
    import glob
    candidates = glob.glob(os.path.join(subj_dir, "**", "*.1D"), recursive=True)
    if not candidates:
        raise FileNotFoundError(f"No .1D file found under {subj_dir}")
    path = candidates[0]
    with open(path) as f:
        header_line = f.readline()
    codes = [int(m) for m in re.findall(r"Mean_(\d+)", header_line)]
    return codes, path


# Pick one real subject to check against
check_row = meta.iloc[0]
check_subj_dir = match_subject_dir(os.path.join(DATA, check_row["site"]), check_row["subject_id"])

codes_in_order, header_path = get_1d_header_codes(check_subj_dir)
print(f"Checked file: {header_path}")
print(f"Header has {len(codes_in_order)} ROI columns (expect {N_ROIS})\n")

# Rebuild the same label_lookup used inside build_aal_roi_names, so we can
# translate the REAL header codes into names, independent of ROI_NAMES.
aal = load_local_aal()
label_lookup = {int(idx): lab for idx, lab in zip(aal.indices, aal.labels)}

mismatches = []
for pos, code in enumerate(codes_in_order):
    expected_name = label_lookup.get(code, f"<unknown code {code}>")
    actual_name = ROI_NAMES[pos]
    status = "OK" if expected_name == actual_name else "MISMATCH"
    if status == "MISMATCH":
        mismatches.append((pos, code, expected_name, actual_name))
    print(f"[{pos+1:>3}] .1D code={code:<6} header-says={expected_name:<25} ROI_NAMES[{pos}]={actual_name:<25} {status}")

print()
if not mismatches:
    print(f"ALL {len(codes_in_order)} POSITIONS MATCH -- ROI_NAMES ordering is verified correct.")
else:
    print(f"{len(mismatches)} MISMATCHES FOUND -- ROI_NAMES ordering does NOT match your .1D column order.")
    print("Do not trust named edges until this is fixed.")


Checked file: /home/siu856622573/Code/ADHD200_AAL_TCs_filtfix (1)/KKI/1018959/snwmrda1018959_session_1_rest_1_aal_TCs.1D
Header has 116 ROI columns (expect 116)

[  1] .1D code=2001   header-says=Precentral_L              ROI_NAMES[0]=Precentral_L              OK
[  2] .1D code=2002   header-says=Precentral_R              ROI_NAMES[1]=Precentral_R              OK
[  3] .1D code=2101   header-says=Frontal_Sup_L             ROI_NAMES[2]=Frontal_Sup_L             OK
[  4] .1D code=2102   header-says=Frontal_Sup_R             ROI_NAMES[3]=Frontal_Sup_R             OK
[  5] .1D code=2111   header-says=Frontal_Sup_Orb_L         ROI_NAMES[4]=Frontal_Sup_Orb_L         OK
[  6] .1D code=2112   header-says=Frontal_Sup_Orb_R         ROI_NAMES[5]=Frontal_Sup_Orb_R         OK
[  7] .1D code=2201   header-says=Frontal_Mid_L             ROI_NAMES[6]=Frontal_Mid_L             OK
[  8] .1D code=2202   header-says=Frontal_Mid_R             ROI_NAMES[7]=Frontal_Mid_R             OK
[  9] .1D code=2211   

In [37]:
def compute_node_mean_features(avg_fc):
    """Node feature prompt (paper Sec. 3.2, X^d -> X^1): for each ROI, the mean
    of its row in the FC/Pearson-correlation matrix (X_FC, Appendix C) --
    i.e. that region's average connectivity strength with the rest of the brain.
    NOT the raw BOLD time-series mean -- X_FC and A_FC are the same underlying
    matrix in the paper, just node-reduced vs. edge-thresholded views of it."""
    n = avg_fc.shape[0]
    # exclude the diagonal (self-correlation = 1.0) so it doesn't inflate every mean
    mask = ~np.eye(n, dtype=bool)
    return np.array([avg_fc[i][mask[i]].mean() for i in range(n)])  # (116,)

In [38]:
def fc_to_edge_list(fc_matrix, keep_ratio=0.3, use_raw_corr=True, ts_for_raw_corr=None):
    n = fc_matrix.shape[0]
    iu, ju = np.triu_indices(n, k=1)
    weights = fc_matrix[iu, ju]

    n_keep = max(1, int(len(weights) * keep_ratio))
    top_idx = np.argsort(-np.abs(weights))[:n_keep]

    edges = [(int(iu[k]) + 1, int(ju[k]) + 1, float(weights[k])) for k in top_idx]

    # Sort by |weight| descending -- biggest magnitude first, smallest last
    edges.sort(key=lambda e: abs(e[2]), reverse=True)

    return edges

def build_named_structure_record(edges, node_means, roi_names, weight_decimals=2, feature_decimals=3):
    """Same content/sorting as build_named_structure_prompt, but returns a dict
    with node/edge as separate lists of strings -- not one concatenated blob --
    so the saved JSON is readable/structured, matching how the edge-only
    version is formatted."""
    node_order = np.argsort(-np.abs(node_means))  # descending by |value|

    node_lines = [
        f"{roi_names[i]}: {node_means[i]:.{feature_decimals}f}"
        for i in node_order
    ]
    edge_lines = [
        f"{roi_names[i - 1]}:{w:.{weight_decimals}f}:{roi_names[j - 1]}"
        for (i, j, w) in edges
    ]

    return {
        "node_features": node_lines,
        "edge_features": edge_lines,
    }


def build_all_named_structure_records(root, meta_df, roi_names, keep_ratio=0.3):
    records = {}
    for _, row in meta_df.iterrows():
        subj_dir = match_subject_dir(os.path.join(root, row["site"]), row["subject_id"])
        if subj_dir is None:
            continue

        avg_fc, n_runs = process_subject(subj_dir)
        if avg_fc is None:
            continue

        node_means = compute_node_mean_features(avg_fc)
        edges = fc_to_edge_list(avg_fc, keep_ratio=keep_ratio)
        records[row["subject_id"]] = build_named_structure_record(edges, node_means, roi_names)

    return records


def structured_record_to_prompt_text(record):
    """Converts the {'node_features': ..., 'edge_features': ...} structure
    back into the single text block your LLM prompt template expects
    ('Node feature:\\n...\\n\\nEdge feature:\\n...'). Use this when building
    the actual prompt -- the JSON on disk stays structured/readable, this is
    just for feeding into build_stage1_prompt's brain_graph_text argument."""
    return (
        "Node feature:\n"
        + "\n".join(record["node_features"])
        + "\n\nEdge feature:\n"
        + "\n".join(record["edge_features"])
    )


named_structure_records_all = build_all_named_structure_records(DATA, meta, roi_names=ROI_NAMES, keep_ratio=0.3)

with open("named_structure_records_0.3_v2.json", "w", encoding="utf-8") as f:
    json.dump(named_structure_records_all, f, indent=2)

print(f"Saved {len(named_structure_records_all)} subjects -> named_structure_records_0.3_v2.json")

_check_sid = next(iter(named_structure_records_all))
_rec = named_structure_records_all[_check_sid]
print(f"\n--- {_check_sid} ---")
print(f"node_features ({len(_rec['node_features'])} entries), first 3: {_rec['node_features'][:3]}")
print(f"edge_features ({len(_rec['edge_features'])} entries), first 3: {_rec['edge_features'][:3]}")


Saved 768 subjects -> named_structure_records_0.3_v2.json

--- 1018959 ---
node_features (116 entries), first 3: ['Cerebelum_4_5_R: 0.045', 'Cerebelum_4_5_L: 0.040', 'Amygdala_L: 0.040']
edge_features (2001 entries), first 3: ['Temporal_Sup_L:0.92:Temporal_Sup_R', 'Rectus_L:0.90:Rectus_R', 'Cingulum_Ant_L:0.87:Cingulum_Ant_R']


## 6. `[i]-x-[j]` structure + node features


In [40]:
import os
import json
import numpy as np


def fc_to_edge_list(fc_matrix, keep_ratio=0.3, use_raw_corr=True, ts_for_raw_corr=None):
    n = fc_matrix.shape[0]
    iu, ju = np.triu_indices(n, k=1)
    weights = fc_matrix[iu, ju]

    n_keep = max(1, int(len(weights) * keep_ratio))
    top_idx = np.argsort(-np.abs(weights))[:n_keep]

    edges = [(int(iu[k]) + 1, int(ju[k]) + 1, float(weights[k])) for k in top_idx]

    # Sort by |weight| descending -- biggest magnitude first, smallest last
    edges.sort(key=lambda e: abs(e[2]), reverse=True)

    return edges


def build_structure_prompt(edges, node_means, weight_decimals=2, feature_decimals=3):
    """edges is already sorted biggest -> smallest by fc_to_edge_list above,
    so the text block preserves that order.
      node_means is sorted here the
    same way, by descending |value|, keeping the original node index as the label."""
    node_order = np.argsort(-np.abs(node_means))   # descending by |value|, same convention as edges

    node_lines = [
        f"[{i+1}]:{node_means[i]:.{feature_decimals}f}"
        for i in node_order
    ]
    edge_lines = [
        f"[{i}]:{w:.{weight_decimals}f}:[{j}]"
        for (i, j, w) in edges
    ]

    prompt = (
        "Node feature:\n"
        + "\n".join(node_lines)
        + "\n\nEdge feature:\n"
        + "\n".join(edge_lines)
    )
    return prompt


def build_all_structure_prompts(root, meta_df, keep_ratio=0.3):
    prompts = {}
    for _, row in meta_df.iterrows():
        subj_dir = match_subject_dir(os.path.join(root, row["site"]), row["subject_id"])
        if subj_dir is None:
            continue

        avg_fc, n_runs = process_subject(subj_dir)
        if avg_fc is None:
            continue

        node_means = compute_node_mean_features(avg_fc)   # <-- now derived from avg_fc
        edges = fc_to_edge_list(avg_fc, keep_ratio=keep_ratio)
        prompts[row["subject_id"]] = build_structure_prompt(edges, node_means)

    return prompts


# --- Save, same as cell 34 ---
OUT_DIR = "structure_prompts_v2"
# os.makedirs(OUT_DIR, exist_ok=True)

structure_prompts_all = build_all_structure_prompts(DATA, meta, keep_ratio=0.3)

with open("structure_prompts_v2.json", "w", encoding="utf-8") as f:
    json.dump(structure_prompts_all, f, indent=2)

print(f"Saved {len(structure_prompts_all)} subjects (edges sorted largest -> smallest |weight|):")
print(f"  - combined JSON        -> structure_prompts_v2.json")

Saved 768 subjects (edges sorted largest -> smallest |weight|):
  - combined JSON        -> structure_prompts_v2.json


### Z-scored edge list

In [41]:
def build_zscored_edge_lists_numeric(X, meta_df, keep_ratio=0.3, weight_decimals=2):
    """For each subject, the top keep_ratio fraction of edges by |cohort z-score|,
    as '[i]:z:[j]' strings using numeric ROI indices only
    Sorted largest -> smallest |z|. Returns {subject_id: ["[i]:z:[j]", ...]}."""
    iu, ju = np.triu_indices(N_ROIS, k=1)   # same pair order as upper_triangle_features

    mean = X.mean(axis=0)
    std = X.std(axis=0) + 1e-8
    Z = (X - mean) / std   # (n_subjects, n_edges) cohort z-scores, same normalization as X

    n_edges_total = Z.shape[1]
    n_keep = max(1, int(n_edges_total * keep_ratio))

    zscored_edges_numeric = {}
    for row_idx, (_, row) in enumerate(meta_df.iterrows()):
        z_row = Z[row_idx]
        top_idx = np.argsort(-np.abs(z_row))[:n_keep]   # descending |z|, biggest first
        edges = [
            f"[{iu[k] + 1}]:{z_row[k]:.{weight_decimals}f}:[{ju[k] + 1}]"
            for k in top_idx
        ]
        zscored_edges_numeric[row["subject_id"]] = edges

    return zscored_edges_numeric


ZSCORE_KEEP_RATIO = 0.3

zscored_edge_lists_numeric = build_zscored_edge_lists_numeric(X, meta, keep_ratio=ZSCORE_KEEP_RATIO)

with open("zscored_edge_lists_numeric_0.3.json", "w") as f:
    json.dump(zscored_edge_lists_numeric, f, indent=2)

print(f"Saved z-scored numeric edge lists (keep_ratio={ZSCORE_KEEP_RATIO}) "
      f"for {len(zscored_edge_lists_numeric)} subjects -> zscored_edge_lists_numeric_0.3.json")

Saved z-scored numeric edge lists (keep_ratio=0.3) for 768 subjects -> zscored_edge_lists_numeric_0.3.json


### Z-scored named edge list (recommended for LoRA training)


In [42]:
def build_named_edge_lists_zscored(X, meta_df, roi_names, top_k=500, weight_decimals=2):
    """X: (n_subjects, n_edges) raw-r upper-triangle features from build_dataset/upper_triangle_features.
    Returns {subject_id: ["RoiA-<z>-RoiB", ...]}, top_k edges per subject by |z|, sorted by |z| desc."""
    iu, ju = np.triu_indices(N_ROIS, k=1)   # same order used by upper_triangle_features


# cohort-wise, not within-subject:
    mean = X.mean(axis=0)
    std = X.std(axis=0) + 1e-8
#  Within-subject:
    # mean = X.mean(axis=1, keepdims=True)
    # std = X.std(axis=1, keepdims=True) + 1e-8
    Z = (X - mean) / std   # (n_subjects, n_edges) cohort z-scores

    named_edges_z = {}
    for row_idx, (_, row) in enumerate(meta_df.iterrows()):
        z_row = Z[row_idx]
        top_idx = np.argsort(-np.abs(z_row))[:top_k]
        edges = [
            f"{roi_names[iu[k]]}-{z_row[k]:.{weight_decimals}f}-{roi_names[ju[k]]}"
            for k in top_idx
        ]
        named_edges_z[row["subject_id"]] = edges

    return named_edges_z


TOP_K_EDGES = 500   # ~100-150 is a reasonable default: enough signal, short enough to train fast

named_edge_lists_zscored = build_named_edge_lists_zscored(X, meta, ROI_NAMES, top_k=TOP_K_EDGES)

with open("named_edge_lists_zscored_top500.json", "w") as f:
    json.dump(named_edge_lists_zscored, f, indent=2)

print(f"Saved z-scored named edge lists for {len(named_edge_lists_zscored)} subjects -> named_edge_lists_zscored_top500.json")
example_sid = meta.iloc[0]["subject_id"]
print(f"\nExample ({example_sid}), top 10 by |z|:")
for e in named_edge_lists_zscored[example_sid]:
    print(" ", e)

Saved z-scored named edge lists for 768 subjects -> named_edge_lists_zscored_top500.json

Example (1018959), top 10 by |z|:
  Cuneus_L-4.21-Pallidum_L
  Insula_R--3.85-Heschl_R
  Parietal_Sup_L-3.81-Vermis_3
  Parietal_Sup_R-3.54-Vermis_3
  Parietal_Inf_L--3.44-Angular_L
  Cuneus_L--3.34-Vermis_3
  Postcentral_R-3.31-Vermis_3
  Hippocampus_R--3.24-Putamen_R
  SupraMarginal_R-3.22-Vermis_3
  Parietal_Inf_R--3.12-Angular_L
  Frontal_Mid_L-3.11-Temporal_Sup_R
  Cuneus_L--3.10-Occipital_Inf_L
  Pallidum_L--3.07-Vermis_3
  ParaHippocampal_R-3.05-Parietal_Sup_L
  ParaHippocampal_R-3.03-Occipital_Mid_L
  Rectus_R--3.01-Vermis_3
  Paracentral_Lobule_R-3.00-Vermis_3
  Temporal_Sup_R--2.98-Temporal_Pole_Mid_R
  Insula_R-2.96-Cuneus_L
  Frontal_Mid_L-2.96-Temporal_Sup_L
  ParaHippocampal_R-2.96-Precuneus_L
  Parietal_Inf_L-2.95-Vermis_3
  Precentral_R--2.94-Frontal_Sup_Orb_L
  Postcentral_R-2.92-Cerebelum_Crus1_L
  Frontal_Sup_L--2.90-Parietal_Inf_R
  Frontal_Mid_L-2.89-Calcarine_R
  Parietal_Sup

In [43]:
def build_named_edge_lists_zscored(X, meta_df, roi_names, keep_ratio=0.3, weight_decimals=2):
    """X: (n_subjects, n_edges) raw-r upper-triangle features from build_dataset/upper_triangle_features.
    Returns {subject_id: ["RoiA-<z>-RoiB", ...]}, top keep_ratio fraction of edges per subject by |z|, sorted by |z| desc."""
    iu, ju = np.triu_indices(N_ROIS, k=1)   # same order used by upper_triangle_features


# cohort-wise, not within-subject:
    mean = X.mean(axis=0)
    std = X.std(axis=0) + 1e-8
#  Within-subject:
    # mean = X.mean(axis=1, keepdims=True)
    # std = X.std(axis=1, keepdims=True) + 1e-8
    Z = (X - mean) / std   # (n_subjects, n_edges) cohort z-scores

    n_edges_total = Z.shape[1]
    n_keep = max(1, int(n_edges_total * keep_ratio))

    named_edges_z = {}
    for row_idx, (_, row) in enumerate(meta_df.iterrows()):
        z_row = Z[row_idx]
        top_idx = np.argsort(-np.abs(z_row))[:n_keep]
        edges = [
            f"{roi_names[iu[k]]}-{z_row[k]:.{weight_decimals}f}-{roi_names[ju[k]]}"
            for k in top_idx
        ]
        named_edges_z[row["subject_id"]] = edges

    return named_edges_z


KEEP_RATIO = 0.3   # top 30% of edges by |z|, same convention as the rest of the pipeline

named_edge_lists_zscored = build_named_edge_lists_zscored(X, meta, ROI_NAMES, keep_ratio=KEEP_RATIO)

with open("named_edge_lists_zscored_0.3.json", "w") as f:
    json.dump(named_edge_lists_zscored, f, indent=2)

print(f"Saved z-scored named edge lists (keep_ratio={KEEP_RATIO}) for {len(named_edge_lists_zscored)} subjects -> named_edge_lists_zscored_0.3.json")
example_sid = meta.iloc[0]["subject_id"]
print(f"\nExample ({example_sid}), top 10 by |z|:")
for e in named_edge_lists_zscored[example_sid][:10]:
    print(" ", e)

Saved z-scored named edge lists (keep_ratio=0.3) for 768 subjects -> named_edge_lists_zscored_0.3.json

Example (1018959), top 10 by |z|:
  Cuneus_L-4.21-Pallidum_L
  Insula_R--3.85-Heschl_R
  Parietal_Sup_L-3.81-Vermis_3
  Parietal_Sup_R-3.54-Vermis_3
  Parietal_Inf_L--3.44-Angular_L
  Cuneus_L--3.34-Vermis_3
  Postcentral_R-3.31-Vermis_3
  Hippocampus_R--3.24-Putamen_R
  SupraMarginal_R-3.22-Vermis_3
  Parietal_Inf_R--3.12-Angular_L


# Positive FC + Top 10%

In [44]:
POSITIVE_CACHE_PATH = "fc_dataset_ADHD_positive.npz"

_raw_npz = np.load("fc_dataset_ADHD.npz", allow_pickle=True)
X_raw = _raw_npz["X"]
meta_positive = pd.DataFrame(_raw_npz["meta"].item())

X_positive = np.where(X_raw > 0, X_raw, 0.0)   # <-- negatives discarded, once, for everything downstream

np.savez_compressed(POSITIVE_CACHE_PATH, X=X_positive, meta=meta_positive.to_dict())

print(f"Saved positive-only cache -> {POSITIVE_CACHE_PATH}")
print("Shape:", X_positive.shape)
print("Fraction of entries zeroed out (were negative):", (X_raw <= 0).mean())
print("Any negative values remaining (should be False):", bool((X_positive < 0).any()))

Saved positive-only cache -> fc_dataset_ADHD_positive.npz
Shape: (768, 6670)
Fraction of entries zeroed out (were negative): 0.5200661388055972
Any negative values remaining (should be False): False


In [45]:
# --- Step 2: everything below reads ONLY from X_positive / meta_positive ---

_iu2, _ju2 = np.triu_indices(N_ROIS, k=1)          # same column order as upper_triangle_features, k=1 = diagonal was never included
_n_pairs = len(_iu2)
_n_keep_top10 = max(1, int(_n_pairs * 0.10))       # 667

# Precompute, for each ROI, which upper-triangle columns belong to its row
# (both "i is the smaller index" and "i is the larger index" cases), so we
# can reconstruct row-means without ever rebuilding the full 116x116 matrix.
_cols_for_node = [np.where((_iu2 == i) | (_ju2 == i))[0] for i in range(N_ROIS)]

# Population stats for the cohort z-score, computed using ONLY the subjects
# who actually have a positive value at each ROI-pair. Zeros (subjects whose
# connection there was negative and got filtered to 0) are excluded from
# that pair's mean/std entirely -- not counted as a real 0 data point.

_pos_mask = X_positive > 0                                  # (n_subjects, n_pairs)
_counts = _pos_mask.sum(axis=0)                              # how many subjects are positive at each pair

_sums = np.where(_pos_mask, X_positive, 0).sum(axis=0)
_pair_mean_pos = np.divide(_sums, _counts, out=np.zeros(_n_pairs), where=_counts > 0)

_sq_sums = np.where(_pos_mask, X_positive ** 2, 0).sum(axis=0)
_var_pos = np.divide(_sq_sums, _counts, out=np.zeros(_n_pairs), where=_counts > 0) - _pair_mean_pos ** 2
_var_pos = np.clip(_var_pos, 0, None)                         # guard tiny negative floating-point noise
_pair_std_pos = np.sqrt(_var_pos) + 1e-8



def node_means_from_cache(row_positive):
    """row_positive: this subject's (6670,) positive-only upper-triangle row.
    Returns (116,) node means -- mean of the POSITIVE entries in each ROI's
    row (zeros from the positive-only filtering do not count as "positive")."""
    means = np.zeros(N_ROIS)
    for i in range(N_ROIS):
        vals = row_positive[_cols_for_node[i]]
        pos = vals[vals > 0]
        means[i] = pos.mean() if pos.size > 0 else 0.0
    return means


def top10_edges_from_cache(row_positive):
    """row_positive: this subject's (6670,) positive-only upper-triangle row.
    Returns top-10%-of-total-pool edges as (i, j, raw_weight) with 1-indexed
    ROI numbers, sorted largest -> smallest. Zeros (i.e. originally-negative
    or genuinely-zero pairs) are never selected, even if fewer than 667
    positive values exist for this subject."""

    positive_mask = row_positive > 0
    pos_idx = np.where(positive_mask)[0]
    pos_weights = row_positive[positive_mask]
    n_keep = min(_n_keep_top10, len(pos_weights))
    order = np.argsort(-pos_weights)[:n_keep]
    top_idx = pos_idx[order]
    edges = [(int(_iu2[k]) + 1, int(_ju2[k]) + 1, float(row_positive[k])) for k in top_idx]
    edges.sort(key=lambda e: e[2], reverse=True)
    return edges


def zscore_edges_from_cache(edges, z_row):
    """Same (i, j) pairs as `edges`, weight replaced by that pair's cohort
    z-score (from the positive-only population), re-sorted largest -> smallest."""
    z_edges = []
    for (i, j, _w) in edges:
        col = _cols_for_node_lookup[(i - 1, j - 1)]
        z_edges.append((i, j, float(z_row[col])))
    z_edges.sort(key=lambda e: e[2], reverse=True)
    return z_edges


_cols_for_node_lookup = {(int(_iu2[k]), int(_ju2[k])): k for k in range(_n_pairs)}


In [46]:
def build_structure_record_numbers(edges, node_means, weight_decimals=2, feature_decimals=3):
    """Missing-piece fix: this was defined earlier in the full notebook but
    got dropped from this trimmed version. Same text convention used
    elsewhere: node lines sorted descending, edge lines pre-sorted by caller."""
    node_order = np.argsort(-np.abs(node_means))
    node_lines = [f"[{i+1}]:{node_means[i]:.{feature_decimals}f}" for i in node_order]
    edge_lines = [f"[{i}]:{w:.{weight_decimals}f}:[{j}]" for (i, j, w) in edges]
    return {"node_features": node_lines, "edge_features": edge_lines}


def build_structure_record_names(edges, node_means, roi_names, weight_decimals=2, feature_decimals=3):
    """Same as above, ROI-name format."""
    node_order = np.argsort(-np.abs(node_means))
    node_lines = [f"{roi_names[i]}: {node_means[i]:.{feature_decimals}f}" for i in node_order]
    edge_lines = [f"{roi_names[i - 1]}:{w:.{weight_decimals}f}:{roi_names[j - 1]}" for (i, j, w) in edges]
    return {"node_features": node_lines, "edge_features": edge_lines}


In [47]:
def build_prompt_text(edges, node_means=None, roi_names=None, weight_decimals=2, feature_decimals=3):
    """Single text block: 'Node feature:\n...\n\nEdge feature:\n...'.
    - node_means=None  -> Node feature section is omitted entirely (edges only).
    - node_means given -> ALL 116 ROIs are listed (node_means_from_cache always
      returns a full-length array; a 0.0 just means that ROI had no positive
      connections, it is still listed, not dropped).
    - roi_names=None   -> numeric [i] labels. roi_names given -> AAL names.
    One function handles both output files (edges / z-score) and both label
    styles (numbers / names) -- nothing else needs to be written twice."""
    if roi_names is None:
        edge_lines = [f"[{i}]:{w:.{weight_decimals}f}:[{j}]" for (i, j, w) in edges]
    else:
        edge_lines = [f"{roi_names[i - 1]}:{w:.{weight_decimals}f}:{roi_names[j - 1]}" for (i, j, w) in edges]

    if node_means is None:
        return "Edge feature:\n" + "\n".join(edge_lines)

    node_order = np.argsort(-np.abs(node_means))  # all 116 ROIs, descending by value
    if roi_names is None:
        node_lines = [f"[{i+1}]:{node_means[i]:.{feature_decimals}f}" for i in node_order]
    else:
        node_lines = [f"{roi_names[i]}: {node_means[i]:.{feature_decimals}f}" for i in node_order]

    return "Node feature:\n" + "\n".join(node_lines) + "\n\nEdge feature:\n" + "\n".join(edge_lines)



In [48]:
# --- Run the positive/top-10% pipeline over all subjects, save the 4 files ---
# Edge-list files: full text block, node features for all 116 ROIs included.
# Z-score files: edges only, no node section (node_means=None).
edges_numbers, edges_names = {}, {}
zscore_numbers, zscore_names = {}, {}

for row_idx, subj_id in enumerate(meta_positive["subject_id"]):
    row_positive = X_positive[row_idx]

    node_means = node_means_from_cache(row_positive)
    edges = top10_edges_from_cache(row_positive)

    edges_numbers[subj_id] = build_prompt_text(edges, node_means)
    edges_names[subj_id] = build_prompt_text(edges, node_means, roi_names=ROI_NAMES)

    z_row = (row_positive - _pair_mean_pos) / _pair_std_pos
    z_edges = zscore_edges_from_cache(edges, z_row)

    zscore_numbers[subj_id] = build_prompt_text(z_edges)                       # no node_means -> no node section
    zscore_names[subj_id] = build_prompt_text(z_edges, roi_names=ROI_NAMES)    # no node_means -> no node section

with open("edge_list_top10pos_numbers.json", "w", encoding="utf-8") as f:
    json.dump(edges_numbers, f, indent=2)
with open("edge_list_top10pos_names.json", "w", encoding="utf-8") as f:
    json.dump(edges_names, f, indent=2)
with open("zscore_top10pos_numbers.json", "w", encoding="utf-8") as f:
    json.dump(zscore_numbers, f, indent=2)
with open("zscore_top10pos_names.json", "w", encoding="utf-8") as f:
    json.dump(zscore_names, f, indent=2)

print(f"Saved {len(edges_numbers)} subjects -> edge_list_top10pos_numbers.json")
print(f"Saved {len(edges_names)} subjects -> edge_list_top10pos_names.json")
print(f"Saved {len(zscore_numbers)} subjects -> zscore_top10pos_numbers.json")
print(f"Saved {len(zscore_names)} subjects -> zscore_top10pos_names.json")

_sid = next(iter(edges_numbers))
print(f"\n--- {_sid} edges (numbers), first 300 chars ---")
print(edges_numbers[_sid][:300])
print(f"\n--- {_sid} zscore (numbers), first 300 chars (should have NO 'Node feature' section) ---")
print(zscore_numbers[_sid][:300])


Saved 768 subjects -> edge_list_top10pos_numbers.json
Saved 768 subjects -> edge_list_top10pos_names.json
Saved 768 subjects -> zscore_top10pos_numbers.json
Saved 768 subjects -> zscore_top10pos_names.json

--- 1018959 edges (numbers), first 300 chars ---
Node feature:
[26]:0.323
[82]:0.298
[81]:0.296
[29]:0.294
[53]:0.293
[59]:0.289
[35]:0.286
[25]:0.284
[36]:0.283
[56]:0.280
[92]:0.278
[88]:0.277
[63]:0.276
[91]:0.275
[24]:0.273
[5]:0.273
[11]:0.268
[79]:0.268
[73]:0.268
[23]:0.265
[87]:0.264
[51]:0.264
[3]:0.264
[52]:0.258
[60]:0.258
[17]:0.255
[6

--- 1018959 zscore (numbers), first 300 chars (should have NO 'Node feature' section) ---
Edge feature:
[60]:4.12:[110]
[45]:4.10:[75]
[59]:4.06:[110]
[58]:3.64:[110]
[70]:3.56:[110]
[56]:3.51:[59]
[40]:3.47:[67]
[58]:3.44:[91]
[60]:3.42:[91]
[59]:3.39:[91]
[70]:3.28:[91]
[65]:3.22:[88]
[53]:3.10:[59]
[34]:3.05:[45]
[65]:3.00:[87]
[52]:2.94:[110]
[40]:2.94:[51]
[64]:2.91:[110]
[19]:2.91:


### LogisticRegression - 10 fold

In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


# X = Fc_dataset_ADHD.npz
mean = X.mean(axis=0)
std = X.std(axis=0) + 1e-8
Z = (X - mean) / std   # cohort z-scored edges, same normalization as named_edge_lists_zscored.json

clf = LogisticRegression(max_iter=1500, class_weight="balanced")
scores = cross_val_score(clf, Z, meta["DX_binary"], cv=10, scoring="accuracy")

print(f"Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")
print(f"Per-fold: {scores.round(3)}")

Accuracy: 0.483 +/- 0.094
Per-fold: [0.455 0.468 0.429 0.494 0.494 0.481 0.26  0.623 0.566 0.566]


## 10. Save all structure prompts to disk

**One `.txt` file per subject** (`structure_prompts/<subject_id>.txt`) - easiest to feed one
  file at a time into an LLM API call.
